# 031 · Batch Normalization

Normalising the *inputs* to a network is standard. Batch norm is the same idea
applied **inside** it — because a layer's outputs are the next layer's inputs.

| Part | What we reproduce |
|---|---|
| A | the two steps: normalise, then scale and shift by learned γ and β |
| B | activation drift across 10 layers: **32% without, 2% with** — about **14× tighter** |
| C | why γ and β must exist — the network has to be able to undo it |
| D | training and inference are different, and small batches are a problem |

Needs `numpy`.

In [ ]:
import numpy as np

N, WIDTH, DEPTH = 512, 128, 10
def relu(z):
    return np.maximum(0.0, z)

## Part A — The two steps

Batch norm sits **between `z` and the activation**. Step one normalises using
**this mini-batch's** statistics, per neuron:

$$\hat{z} = \frac{z - \mu}{\sqrt{\sigma^2 + \varepsilon}}$$

Step two scales and shifts with **learned** parameters:

$$z_{\text{out}} = \gamma \hat{z} + \beta$$

In [ ]:
def batch_norm(z, gamma=1.0, beta=0.0, eps=1e-8):
    mu = z.mean(axis=0)                  # per NEURON, across the batch
    var = z.var(axis=0)
    z_hat = (z - mu) / np.sqrt(var + eps)
    return gamma * z_hat + beta


rng = np.random.default_rng(0)
z = rng.normal(loc=4.0, scale=3.0, size=(256, 5))    # a badly-scaled layer

print("before:  mean %.3f  std %.3f" % (z.mean(), z.std()))
out = batch_norm(z)
print("after :  mean %.3f  std %.3f" % (out.mean(), out.std()))

assert abs(out.mean()) < 1e-6 and abs(out.std() - 1) < 1e-3

In [ ]:
# Note the axis. It normalises each neuron separately, not the layer as a whole.
z = np.c_[rng.normal(0, 1, 256), rng.normal(50, 10, 256)]   # two very different neurons
out = batch_norm(z)
print("column means before:", np.round(z.mean(axis=0), 2))
print("column stds  before:", np.round(z.std(axis=0), 2))
print("column means after :", np.round(out.mean(axis=0), 6))
print("column stds  after :", np.round(out.std(axis=0), 6))
print("\nPer layer (optional on each), and per neuron within it.")

## Part B — What it does to a deep stack

Ten He-initialised ReLU layers, with and without normalising each
pre-activation.

He initialisation already prevents the collapse of lesson 029, so this is **not
a rescue**. It is about how tightly the distribution is *held* — which is what
"internal covariate shift" refers to.

In [ ]:
def propagate(use_bn, seed=1):
    rng = np.random.default_rng(seed)
    a = rng.standard_normal((N, WIDTH))
    means, stds = [], []
    for _ in range(DEPTH):
        W = rng.standard_normal((a.shape[1], WIDTH)) * (2 / a.shape[1]) ** 0.5   # He
        z = a @ W
        if use_bn:
            z = (z - z.mean(axis=0)) / (z.std(axis=0) + 1e-8)
        a = relu(z)
        means.append(float(a.mean()))
        stds.append(float(a.std()))
    return means, stds


m_off, s_off = propagate(False)
m_on, s_on = propagate(True)

print(f"{'layer':<7}{'no BN mean':>12}{'no BN std':>11}{'BN mean':>11}{'BN std':>9}")
for i in range(DEPTH):
    print(f"{i+1:<7}{m_off[i]:>12.3f}{s_off[i]:>11.3f}{m_on[i]:>11.3f}{s_on[i]:>9.3f}")

In [ ]:
def spread(v):
    return (max(v) - min(v)) / min(v)

print(f"without BN: std {min(s_off):.3f}-{max(s_off):.3f}  "
      f"({spread(s_off):.0%} variation across depth)")
print(f"with BN   : std {min(s_on):.3f}-{max(s_on):.3f}  "
      f"({spread(s_on):.0%} variation across depth)")
print(f"\n-> batch norm holds the distribution about "
      f"{spread(s_off) / spread(s_on):.0f}x tighter")

assert spread(s_off) / spread(s_on) > 8

Note what the numbers *do not* say. Without batch norm the network is perfectly
trainable — He initialisation saw to that. What batch norm buys is a layer whose
input distribution stops **moving** while the weights below it change, which is
what lets you raise the learning rate without the training becoming unstable.

## Part C — Why γ and β have to exist

If normalising to mean 0 / std 1 were always right, you would just do it. It is
not always right, and the clearest case is a saturating activation.

In [ ]:
z = np.random.default_rng(5).normal(0, 1, 100_000)

# Forced to mean 0, std 1, tanh only ever sees its near-linear middle.
print("with plain normalisation (gamma=1, beta=0):")
print(f"  |tanh(z)| > 0.9 for {np.mean(np.abs(np.tanh(z)) > 0.9):.1%} of inputs")

# gamma lets the network widen the distribution again if it wants to.
for gamma in (1.0, 2.0, 4.0):
    a = np.tanh(gamma * z)
    print(f"  gamma = {gamma}: |tanh| > 0.9 for {np.mean(np.abs(a) > 0.9):5.1%}")

print("\nWith gamma fixed at 1 the network is CONFINED to tanh's linear region,")
print("and a stack of near-linear layers is just one linear layer.")
print("gamma and beta are learned, so the network can undo the normalisation")
print("wherever undoing it is the right thing to do.")

In [ ]:
# In the limit, gamma and beta can restore the original distribution exactly.
rng = np.random.default_rng(2)
z = rng.normal(7.0, 2.5, (1000, 1))

gamma, beta = z.std(), z.mean()          # what the network would have to learn
restored = batch_norm(z, gamma, beta)

print(f"original : mean {z.mean():.3f}  std {z.std():.3f}")
print(f"restored : mean {restored.mean():.3f}  std {restored.std():.3f}")
assert np.allclose(restored, z, atol=1e-3)
print("\nIdentity is inside the hypothesis space. Batch norm can never make")
print("the network strictly less expressive - it can only be turned off.")

## Part D — Training and inference are not the same

At training time the statistics come from the mini-batch. **At inference there is
no batch** — you might be classifying one image. So the layer keeps *running
averages* of μ and σ during training and uses those instead.

Which means batch norm behaves differently in the two modes. That is a real
source of bugs.

In [ ]:
rng = np.random.default_rng(9)
running_mu, running_var, momentum = 0.0, 1.0, 0.9

for step in range(200):                       # "training"
    batch = rng.normal(3.0, 2.0, 64)
    running_mu = momentum * running_mu + (1 - momentum) * batch.mean()
    running_var = momentum * running_var + (1 - momentum) * batch.var()

print(f"true mean 3.0, true var 4.0")
print(f"running estimates: mean {running_mu:.3f}, var {running_var:.3f}")
assert abs(running_mu - 3.0) < 0.5

In [ ]:
# And the small-batch problem: batch statistics get noisy fast.
rng = np.random.default_rng(11)
print("estimating a mean of 0.0 and std of 1.0 from one batch:\n")
print(f"{'batch size':>11}{'mean spread':>14}{'std spread':>12}")
for bs in (256, 64, 16, 4, 2):
    means = [rng.normal(0, 1, bs).mean() for _ in range(2000)]
    stds = [rng.normal(0, 1, bs).std() for _ in range(2000)]
    print(f"{bs:>11}{np.std(means):>14.3f}{np.std(stds):>12.3f}")

print("\nAt batch size 2 the 'normalisation' is mostly noise. This is why batch")
print("norm works poorly with very small batches - and why layer norm, which")
print("normalises across features instead, is what transformers use.")

## What to take away

- **Batch norm is input normalisation applied inside the network** — a layer's
  outputs are the next layer's inputs.
- **Internal covariate shift:** the weights change every update, so each layer's
  input distribution keeps moving.
- Applied with **mini-batch** GD, **per layer** (optional on each), **per neuron**.
- Inserted **between `z` and the activation**.
- **Step 1:** `ẑ = (z − μ)/√(σ² + ε)` using **this mini-batch's** statistics.
- **Step 2:** `z_out = γẑ + β`, with γ and β **learned** — so the network can
  undo it, which matters because mean 0 / std 1 would confine tanh and sigmoid
  to their linear region.
- **At inference there is no batch** — running averages are used instead, so
  behaviour differs from training.
- **It works poorly with very small batches**, where batch statistics are noise.
- Measured: activation std varied **32% without, 2% with** — about **14× tighter**.

## Exercises

1. Rerun Part B with a bad initialisation (×0.01) instead of He. Does batch norm
   rescue it? Should the answer change how you think about what batch norm is for?
2. Batch norm is supposed to allow larger learning rates. Train a small network
   on any dataset at `lr = 0.5` with and without it, and see which diverges.
3. Where exactly should batch norm go — before or after the activation? Try both
   in Part B and compare. Which does Keras do by default?
4. Implement the backward pass of batch norm by hand and check it against a
   numerical gradient. It is more involved than it looks, because μ and σ both
   depend on every element of the batch.
5. Implement **layer norm** (normalise across features, per sample) and rerun
   Part D's small-batch table. Confirm it does not degrade with batch size, and
   explain why that makes it the right choice for transformers.